# Getting started

> *What is the list of machine readable file URLs that represent the Anthem PPO network in New York state?*
> 
> *Your output should be the list of machine readable file URLs corresponding to Anthem's PPO in New York state.*

We want to collect a list of **index files** referenced by *this* index file which correspond to Anthem.

1. Use `json_stream` to iterate over the JSON file
2. **What corresponds to Anthem?** If any of the entries under `reporting_structure` mention Anthem in (1) the `plan_name`, the `issuer_name`, or in the list of files referenced by the `ein`, count that as Anthem. Then, add every URL to the set.
3. **Avoid dupes**: Maintain a `list` of URLs. To avoid dupes, maintain a `set` (with the queryparams stripped) in parallel. (We want a copy of these queryparams, since we need them to actually load the URL, but we expect these can differ for different instances of the same URL.)

In [2]:
# first, extract the json file 
!gzip -c -d 2026-02-01_anthem_index.json.gz > anthem_index.json

In [3]:
# Add dependencies as needed to the `pyproject.toml`.
!uv add pandas
!uv add json_stream

Resolved 13 packages in 4ms
Audited 11 packages in 2ms
Resolved 13 packages in 4ms
Audited 11 packages in 0.30ms


In [55]:
import json
import os, sys
import json_stream
import httpx
import gzip
import time

In [21]:
# Utility functions

def tst( ts_json_object):
    # Utility function since I'll be using this a lot
    return json_stream.to_standard_types(ts_json_object)

def ein_lookup(ein):
    req = httpx.get(f"https://antm-pt-prod-dataz-nogbd-nophi-us-east1.s3.amazonaws.com/anthem/{ein}.json")
    if req.status_code == 200:
        return json.loads(req.text)

def _get_json_gz(url):
    return json.loads(gzip.decompress(httpx.get(url).content))

def _collapse(string: str):
    return string.lower().strip().replace(' ','')


# First pass

## First, let's build our tools, and test it on the first set in the reporting structure

In [9]:
ff = open("anthem_index.json", "r")
index = json_stream.load(ff)
eg = tst(index['reporting_structure'][0])

In [39]:
url_set = set()
url_list = []

def _strip_url(url):
    if "?" in url:
        return url.split("?")[0]
    else:
        return url

assert _strip_url(
    "https://anthembcca.mrf.bcbs.com/2026-02_800_72A0_in-network-rates_02_of_02.json.gz?&Expires=1774274448&Signature=xGdaNAX6qC0ls20iJPBsIB2Mv~aiyCNfq5juNJMKZtK9J2H7BG7TPmsVq2QwNPEIkEw0X4Srpnfflq5N2nd1CI3oaSESVHF-FJJjXHFBcjasaUs9Ss1UIcUoAAsz42U7h8t2dO31XGXRsAVadiVnBrTEeOS6RBpTQ~TmTXkS2SCtzfhg~gkyx~Xm0766BsFhzjncluYdJX4IEcwHJc0T5l1A0QpGPO2pIuzkEPINjAp76Z8AqRJHFgXBlugzVeSeaePAQfqhPs7-15y8yQDiXLlaeG2mza0Rc1YRUrmKehLTRkrcDmRd11ZYs5nJJKgb4~I0O7IbjX8OpTZn-dO4iA__&Key-Pair-Id=K27TQMT39R1C8A"
) == "https://anthembcca.mrf.bcbs.com/2026-02_800_72A0_in-network-rates_02_of_02.json.gz"

def _add_url(url):
    stripped = _strip_url(url)
    if not stripped in url_set:
        url_set.add(stripped)
        url_list.append(url)



In [40]:
def _is_ein_anthem(ein):
    'anthem' in _collapse(str(ein_lookup(ein)))

def is_plan_anthem(plan):
    # check first for 'anthem' in plan name, issuer name, etc
    for key in ['plan_name', 'issuer_name', 'plan_sponsor_name']:
        if 'anthem' in _collapse(plan.get(key, '')):
            return True
    
    # failing that, look up the EIN and check for references there
    if plan.get('plan_id_type', '') == 'EIN':
        return _is_ein_anthem(plan.get('plan_id',0))
    
    return False

assert is_plan_anthem(eg['reporting_plans'][0])

In [52]:
# Example of how to handle one entry (e.g. `index['reporting_structure'][ii]`)

def collect_urls_from_one_reporting_structure(reporting_structure):
    is_anthem = False
    
    for plan in reporting_structure['reporting_plans']:
        if is_plan_anthem(plan):
            is_anthem = True
            break
    
    if is_anthem:
        # one of the entries are anthem, so let's add all of them
        for entry in reporting_structure['in_network_files']:
            _add_url(entry.get('location',''))


In [53]:
# Example "from scratch" on the first entry

ff = open("anthem_index.json", "r")
index = json_stream.load(ff)


url_set = set()
url_list = []

collect_urls_from_one_reporting_structure(
    tst(index['reporting_structure'][0])
)

len(url_list), len(url_set)

(280, 280)

- `ff = open("index.json", "r"); index = json_stream.load(ff)` to re-open the file (you can only move forward, so we will need to restart eventually!)
- `tst(ts_json_object)`: Utility function to get a proper object from a transient json stream
- `ein_lookup(ein)`: Int to dict (from json) for a given EIN
- `_get_json_gz(url)`: Download a `.gz.json` from a URL, uncompress it, read into a dict

- `_strip_url(url)`
- `url_set` and `url_list` with utility setter `_add_url(url)`

- `_is_ein_anthem(ein)` - For an `ein`, look it up, and report if `anthem` is in the list of URLs
- `is_plan_anthem`: Takes in a `reporting_plan` (`{'plan_name':, 'plan_id_type':, ...'}`

- `_strip_url`
- `url_set`, `url_list`, `_add_url`

- `collect_urls_from_one_reporting_structure(structure)`

# Repeat on the entirety of the data

In [64]:
ff = open("anthem_index.json", "r")

index = json_stream.load(ff)
#eg = tst(index['reporting_structure'][0])

url_set = set()
url_list = []

start = time.time()

for ii, reporting_structure in enumerate(index['reporting_structure']):
    if ii % 1000 == 0:
        print(f"index {ii:> 4}")
    collect_urls_from_one_reporting_structure(tst(reporting_structure))

end = time.time()

print(f"Done in {round(end-start)} seconds")

index    0
index  1000
index  2000
index  3000
index  4000
index  5000
index  6000
index  7000
index  8000
index  9000
index  10000
index  11000
index  12000
index  13000
index  14000
index  15000
index  16000
index  17000
index  18000
index  19000
index  20000
index  21000
index  22000
index  23000
index  24000
index  25000
index  26000
index  27000
index  28000
index  29000
index  30000
index  31000
index  32000
index  33000
index  34000
index  35000
index  36000
index  37000
index  38000
index  39000
index  40000
index  41000
index  42000
index  43000
index  44000
index  45000
index  46000
index  47000
index  48000
index  49000
index  50000
index  51000
index  52000
index  53000
index  54000
index  55000
index  56000
index  57000
index  58000
index  59000
index  60000
index  61000
index  62000
index  63000
index  64000
index  65000
index  66000
index  67000
index  68000
index  69000
index  70000
index  71000
index  72000
index  73000
index  74000
index  75000
index  76000
index  770

In [69]:
with open("url_list.txt", "w") as ff:
    for url in url_list:
        ff.writelines(url)

In [67]:
with open("url_set.txt", "w") as ff:
    ff.writelines(list(url_set))